# NullVector Phase 03 LLM Gateway Cookbook
This cookbook demonstrates deterministic noop, mocked LiteLLM, mocked OpenAI Responses, direct `invoke_many()` batching, an optional experimental LiteLLM + Moonshot transport probe, failure handling, and gateway-backed repair examples.


### Environment
- The cookbook is deterministic by default.
- Mocked noop, mocked LiteLLM, mocked OpenAI strict, repair, and `invoke_many()` examples should always run.
- `invoke_many()` is a gateway-side ordered batch facade that preserves per-item audits and typed failures; it is not provider-native bulk submission.
- The Moonshot section is an experimental transport-compatible probe. It is skipped by default unless `ENABLE_MOONSHOT_LIVE=1`.
- The OpenAI strict example remains mocked by default to preserve deterministic CI behavior.


In [16]:
# environment setup
from pathlib import Path

REPO_ROOT = Path.cwd()
COOKBOOK_ROOT = REPO_ROOT / "notebooks" / "_artifacts" / "phase03-cookbook"
COOKBOOK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"cookbook_root={COOKBOOK_ROOT}")

cookbook_root=/home/pruthvi/projects/NullVector/notebooks/notebooks/_artifacts/phase03-cookbook


In [ ]:
# imports
import json
import os

from pydantic import BaseModel

from nullvector.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayError,
    GatewayRequest,
    GatewayService,
    LLMMessage,
    LLMRole,
    NoopProviderAdapter,
    NoopScriptedResponse,
    StructuredOutputMode,
)

In [ ]:
# configuration
class EchoResponse(BaseModel):
    message: str

os.environ.setdefault("PHASE03_COOKBOOK_OPENAI_API_KEY", "cookbook-mock-key")
os.environ["PHASE03_COOKBOOK_MOONSHOT_API_KEY"] = (
    "sk-F92EGVHyDNEz9yOhNZITXXPP7ln7G1brTu7lbXi0CAHzqDe5"
)
os.environ["PHASE03_COOKBOOK_MOONSHOT_BASE_URL"] = "https://api.moonshot.ai/v1"

ENABLE_MOONSHOT_LIVE = os.getenv("ENABLE_MOONSHOT_LIVE", "0") == "1"

noop_gateway = GatewayService(
    GatewayConfig(
        default_model="noop-model",
        audit=GatewayAuditConfig(persist_root=str(COOKBOOK_ROOT / "noop-audit")),
    ),
    provider_adapter=NoopProviderAdapter(
        {
            "cookbook-noop": NoopScriptedResponse(
                output_json={"message": "hello from noop cookbook"}
            ),
            "cookbook-invalid": NoopScriptedResponse(output_json={"wrong": "shape"}),
            "cookbook-batch-a": NoopScriptedResponse(
                output_json={"message": "hello from invoke_many batch item A"}
            ),
            "cookbook-batch-b": NoopScriptedResponse(
                output_json={"message": "hello from invoke_many batch item B"}
            ),
        }
    ),
)

litellm_gateway = GatewayService(
    GatewayConfig(
        default_model="openai/gpt-4.1-mini",
        audit=GatewayAuditConfig(persist_root=str(COOKBOOK_ROOT / "litellm-audit")),
        structured_output_mode_preference=StructuredOutputMode.TRANSPORT_COMPATIBLE,
    ),
    provider_adapter=NoopProviderAdapter(
        {
            "cookbook-litellm": NoopScriptedResponse(
                output_json={"message": "hello from litellm cookbook"}
            ),
        }
    ),
)

moonshot_gateway = GatewayService(
    GatewayConfig(
        default_model="moonshot/kimi-k2.5",
        audit=GatewayAuditConfig(persist_root=str(COOKBOOK_ROOT / "moonshot-audit")),
        structured_output_mode_preference=StructuredOutputMode.TRANSPORT_COMPATIBLE,
        timeout_seconds=10.0,
    ),
    provider_adapter=NoopProviderAdapter(
        {
            "cookbook-moonshot-live-probe": NoopScriptedResponse(
                output_json={"message": "hello from moonshot cookbook"}
            ),
        }
    ),
)

openai_noop = NoopProviderAdapter(
    {
        "cookbook-openai": NoopScriptedResponse(
            output_json={"message": "hello from openai cookbook"}
        ),
    }
)
openai_gateway = GatewayService(
    GatewayConfig(
        default_model="gpt-4.1-mini",
        audit=GatewayAuditConfig(persist_root=str(COOKBOOK_ROOT / "openai-audit")),
    ),
    provider_adapter=openai_noop,
)


In [ ]:
# execution
noop_success = noop_gateway.invoke(
    GatewayRequest[EchoResponse](
        operation_name="cookbook-noop",
        messages=(LLMMessage(role=LLMRole.USER, content="return a noop greeting"),),
        response_model=EchoResponse,
        idempotency_key="cookbook-noop",
    )
)

litellm_success = litellm_gateway.invoke(
    GatewayRequest[EchoResponse](
        operation_name="cookbook-litellm",
        messages=(LLMMessage(role=LLMRole.USER, content="return a transport-compatible greeting"),),
        response_model=EchoResponse,
        idempotency_key="cookbook-litellm",
        structured_output_mode=StructuredOutputMode.TRANSPORT_COMPATIBLE,
    )
)

openai_success = openai_gateway.invoke(
    GatewayRequest[EchoResponse](
        operation_name="cookbook-openai",
        messages=(LLMMessage(role=LLMRole.USER, content="return a strict greeting"),),
        response_model=EchoResponse,
        idempotency_key="cookbook-openai",
    )
)
batch_successes = noop_gateway.invoke_many(
    (
        GatewayRequest[EchoResponse](
            operation_name="cookbook-batch-a",
            messages=(LLMMessage(role=LLMRole.USER, content="return batch item A"),),
            response_model=EchoResponse,
            idempotency_key="cookbook-batch-a",
        ),
        GatewayRequest[EchoResponse](
            operation_name="cookbook-batch-b",
            messages=(LLMMessage(role=LLMRole.USER, content="return batch item B"),),
            response_model=EchoResponse,
            idempotency_key="cookbook-batch-b",
        ),
    ),
    max_workers=2,
)


In [20]:
# execution
validation_error_summary = None
try:
    noop_gateway.invoke(
        GatewayRequest[EchoResponse](
            operation_name="cookbook-invalid",
            messages=(LLMMessage(role=LLMRole.USER, content="return an invalid payload"),),
            response_model=EchoResponse,
            idempotency_key="cookbook-invalid",
        )
    )
except Exception as exc:
    validation_error_summary = {
        "type": type(exc).__name__,
        "message": str(exc),
        "audit_path": getattr(exc, "audit_path", None),
    }

> **Note:** Repair functionality was removed during the VLM/LLM architecture pivot.


### Experimental Moonshot Transport Probe
This section is optional and is not Phase 03 acceptance evidence. It exercises the current LiteLLM responses-style path against Moonshot and records either a typed success or a typed failure audit artifact when enabled.


In [22]:
# execution: experimental Moonshot transport probe
moonshot_live_probe = {
    "status": "skipped",
    "reason": "Set ENABLE_MOONSHOT_LIVE=1 to run the optional live Moonshot transport probe.",
    "assurance": "transport_compatible",
    "audit_path": None,
}

if ENABLE_MOONSHOT_LIVE:
    try:
        moonshot_success = moonshot_gateway.invoke(
            GatewayRequest[EchoResponse](
                operation_name="cookbook-moonshot-live-probe",
                messages=(
                    LLMMessage(
                        role=LLMRole.USER,
                        content="Return JSON with a single message field describing the Moonshot live cookbook probe path.",
                    ),
                ),
                response_model=EchoResponse,
                idempotency_key="cookbook-moonshot-live-probe",
                structured_output_mode=StructuredOutputMode.TRANSPORT_COMPATIBLE,
                temperature=1.0,
                max_output_tokens=80,
            )
        )
        moonshot_live_probe = {
            "status": "success",
            "message": moonshot_success.output.message,
            "assurance": moonshot_success.assurance_mode.value,
            "attempts": len(moonshot_success.attempts),
            "usage": (
                moonshot_success.usage.model_dump(mode="json")
                if moonshot_success.usage is not None
                else None
            ),
            "audit_path": moonshot_success.audit_path,
        }
    except GatewayError as exc:
        moonshot_live_probe = {
            "status": "failure",
            "category": exc.failure.category.value,
            "message": str(exc),
            "provider": exc.failure.provider_name,
            "retryable": exc.failure.retryable,
            "audit_path": exc.audit_path,
        }
    except Exception as exc:
        moonshot_live_probe = {
            "status": "failure",
            "category": "unexpected_exception",
            "message": str(exc),
            "type": type(exc).__name__,
            "audit_path": None,
        }

In [23]:
# inspect results
def load_probe_audit_excerpt(probe_summary):
    audit_path = probe_summary.get("audit_path")
    if not audit_path:
        return None
    path = Path(audit_path)
    if not path.exists():
        return {"missing_path": str(path)}
    payload = json.loads(path.read_text(encoding="utf-8"))
    failure = payload.get("failure") or {}
    response_payload = payload.get("response_payload") or {}
    return {
        "provider_name": payload.get("provider_name"),
        "assurance_mode": payload.get("assurance_mode"),
        "failure_category": failure.get("category"),
        "provider_response_id": failure.get("provider_response_id"),
        "response_object": response_payload.get("object"),
        "response_status": response_payload.get("status"),
    }


results = {
    "noop": {"message": noop_success.output.message, "audit_path": noop_success.audit_path},
    "litellm": {
        "message": litellm_success.output.message,
        "assurance": litellm_success.assurance_mode.value,
        "audit_path": litellm_success.audit_path,
    },
    "openai": {
        "message": openai_success.output.message,
        "assurance": openai_success.assurance_mode.value,
        "audit_path": openai_success.audit_path,
    },
    "batch_invoke_many": {
        "messages": [success.output.message for success in batch_successes],
        "audit_paths": [success.audit_path for success in batch_successes],
    },
    "moonshot_live_probe": moonshot_live_probe,
    "moonshot_live_probe_audit_excerpt": load_probe_audit_excerpt(moonshot_live_probe),
    "validation_error": validation_error_summary,
    "audit_files": sorted(
        str(path.relative_to(COOKBOOK_ROOT)) for path in COOKBOOK_ROOT.rglob("*.json")
    ),
}
print(json.dumps(results, indent=2, sort_keys=True))

{
  "audit_files": [
    "litellm-audit/cookbook-litellm.json",
    "moonshot-audit/cookbook-moonshot-live.json",
    "noop-audit/cookbook-invalid.json",
    "noop-audit/cookbook-noop.json",
    "openai-audit/cookbook-openai.json",
    "repair-audit/cookbook-repair.json"
  ],
  "litellm": {
    "assurance": "transport_compatible",
    "audit_path": "/home/pruthvi/projects/NullVector/notebooks/notebooks/_artifacts/phase03-cookbook/litellm-audit/cookbook-litellm.json",
    "message": "hello from litellm cookbook"
  },
  "moonshot_live_probe": {
    "assurance": "transport_compatible",
    "audit_path": null,
    "reason": "Set ENABLE_MOONSHOT_LIVE=1 to run the optional live Moonshot transport probe.",
    "status": "skipped"
  },
  "moonshot_live_probe_audit_excerpt": null,
  "noop": {
    "audit_path": "/home/pruthvi/projects/NullVector/notebooks/notebooks/_artifacts/phase03-cookbook/noop-audit/cookbook-noop.json",
    "message": "hello from noop cookbook"
  },
  "openai": {
    "assura

### Known Limitations
- LiteLLM examples here remain transport-compatible only.
- The OpenAI strict examples are mocked by default to keep CI and notebook execution deterministic.
- The Moonshot probe is skipped by default and should not be treated as validated structured-output support for the current LiteLLM responses-style path.
- When enabled, the Moonshot probe may return a typed gateway failure instead of a schema-valid model, and that outcome should be inspected through the persisted audit artifact.
- The embedded Moonshot key should be treated as disposable and rotated after use.
- `invoke_many()` batches gateway dispatch only; it does not submit a provider-native bulk job.
